In [30]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

In [31]:
df_data = pd.read_csv("./dataset_phishing.csv")
print(df_data.shape)
df_data.head()

(11430, 89)


,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate


In [32]:
df_data['status'].value_counts()

status
legitimate    5715
phishing      5715
Name: count, dtype: int64

In [33]:
df_data['target'] = pd.get_dummies(df_data['status'])['legitimate'].astype(int)
df_data.drop('status', axis=1, inplace=True)
df_data[['url', 'target']].head()


,url,target
0,http://www.crestonwood.com/router.php,1
1,http://shadetreetechnology.com/V4/validation/a...,0
2,https://support-appleld.com.secureupdate.duila...,0
3,http://rgipt.ac.in,1
4,http://www.iracing.com/tracks/gateway-motorspo...,1


In [34]:
from sklearn.model_selection import train_test_split

X = df_data.iloc[:, 1: -1] # url를 빼고 나머지 들고오기
y = df_data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2 ,stratify=y)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(9144, 87) (9144,)
(2286, 87) (2286,)


In [35]:
print(y_train.values, y_test.values)
print(y_train.shape, y_test.shape)

[0 0 0 ... 1 0 1] [1 1 1 ... 1 0 0]
(9144,) (2286,)


In [36]:
print(type(y_train)) # Series 형태인데
print(type(y_train.values)) # values를 하면 numpy의 ndarray 형태이다

<class 'pandas.core.series.Series'>
<class 'numpy.ndarray'>


In [37]:
std_scaler = StandardScaler()
std_scaler.fit(X_train)
X_train_tensor = torch.from_numpy(std_scaler.transform(X_train)).float()
X_test_tensor = torch.from_numpy(std_scaler.transform(X_test)).float()
y_train_tensor = torch.from_numpy(y_train.values).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test.values).float().unsqueeze(1)


In [38]:
print(y_train_tensor.values, y_test_tensor.values)
print(y_train_tensor.shape, y_test_tensor.size())

<built-in method values of Tensor object at 0x1447b79d0> <built-in method values of Tensor object at 0x1447aa120>
torch.Size([9144, 1]) torch.Size([2286, 1])


In [39]:
nb_epochs = 1000 # 1000 epoch 실행 설정
minibatch_size = 256 # Mini-batch 사이즈는 256개로 설정

In [40]:
class FunModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear_layers = nn.Sequential(
            nn.Linear(input_dim, 200),
            nn.LeakyReLU(0.1),
            nn.Linear(200, 100),
            nn.LeakyReLU(0.1),
            nn.Linear(100, 20),
            nn.LeakyReLU(0.1),
            nn.Linear(20, 5),
            nn.LeakyReLU(0.1),
            nn.Linear(5, output_dim),
            nn.Sigmoid() # 반드시 BCELoss() 사용을 위해 마지막은 Sigmoid() 함수를 사용해야 합니다.
        )
    def forward(self, x):
        y = self.linear_layers(x)
        return y

In [41]:
X_test_tensor.size()

torch.Size([2286, 87])

In [42]:
input_dim = X_train_tensor.size(-1)
output_dim = y_train_tensor.size(-1)
print(input_dim, output_dim)
model = FunModel(input_dim, output_dim)
loss_func = nn.BCELoss() # 반드시 BCELoss 사용을 위해 마지막은 Sigmoid() 함수를 사용해야 함!!!
optimizer = torch.optim.Adam(model.parameters()) # Adam, learning rate 필요없음

87 1


In [43]:
for index in range(nb_epochs):
    indices = torch.randperm(X_train_tensor.size(0))
    
    x_batch_list = torch.index_select(X_train_tensor, 0, index=indices)
    y_batch_list = torch.index_select(y_train_tensor, 0, index=indices)
    x_batch_list = x_batch_list.split(minibatch_size, 0)
    y_batch_list = y_batch_list.split(minibatch_size, 0)
    
    epoch_loss = list()
    for x_minibatch, y_minibatch in zip(x_batch_list, y_batch_list):
        y_minibatch_pred = model(x_minibatch)
        
        loss = loss_func(y_minibatch_pred, y_minibatch)
        epoch_loss.append(loss)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    if(index % 100) == 0:
        print(index, sum(epoch_loss)/len(epoch_loss))

0 tensor(0.5397, grad_fn=<DivBackward0>)
100 tensor(0.0109, grad_fn=<DivBackward0>)
200 tensor(0.0109, grad_fn=<DivBackward0>)
300 tensor(0.0109, grad_fn=<DivBackward0>)
400 tensor(5.0211e-05, grad_fn=<DivBackward0>)
500 tensor(6.0019e-06, grad_fn=<DivBackward0>)
600 tensor(4.6759e-06, grad_fn=<DivBackward0>)
700 tensor(5.2707e-07, grad_fn=<DivBackward0>)
800 tensor(5.8131e-08, grad_fn=<DivBackward0>)
900 tensor(7.3467e-09, grad_fn=<DivBackward0>)


In [44]:
y_pred_list = []
model.eval()
with torch.no_grad():
    y_test_pred_sigmoid = model(X_test_tensor)
    y_test_pred = torch.round(y_test_pred_sigmoid)

**mini-batch size 기반 예측**

In [55]:
y_pred_list = list()
x_test_batch_list = X_test_tensor.split(minibatch_size, 0)
model.eval()
with torch.no_grad():
    for x_minibatch in x_test_batch_list:
        y_test_pred_sigmoid = model(x_minibatch)
        y_test_pred = torch.round(y_test_pred_sigmoid)
        y_pred_list.extend(y_test_pred.squeeze().detach().tolist())

y_pred_list = torch.tensor(y_pred_list).unsqueeze(1)

In [56]:
print(y_test_tensor.size(), y_pred_list.shape)

torch.Size([2286, 1]) torch.Size([2286, 1])


### Classification 기본 메트릭

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

print(confusion_matrix(y_test_tensor, y_pred_list))
print("Precision: " + str(precision_score(y_test_tensor, y_pred_list)))
print("Recall: "+ str(recall_score(y_test_tensor, y_pred_list)))
print("F1 SCORE: " + str(f1_score(y_test_tensor, y_pred_list)))  # Precision과 Recall 이 적절할 경우 F1 SCORE이 나온다.


[[1099   44]
 [  46 1097]]
Precision: 0.9614373356704645
Recall: 0.9597550306211724
F1 SCORE: 0.9605954465849387
